# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python-api/) library.

### Dataset Source
 - Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# For visualization in later steps
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets in the dataset by their @id and name
record_sets = list(dataset.metadata.record_sets)

print("Available record sets:")
for rs in record_sets:
    print(f"  - @id: {rs.id} | name: {rs.name}")

# For each record set, list fields (columns)
for rs in record_sets:
    print(f"\nRecord set '@id': {rs.id} ({rs.name})")
    print("Fields (columns):")
    for field in rs.fields:
        print(f"  - @id: {field.id} | name: {field.name} | data_type: {field.data_type}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
All record set and field references are made using their `@id` fields.

In [ ]:
# Choose record sets by their @id (from previous cell)
# For this dataset, main data is usually in one or more core record sets. We'll print them above for user selection.

# Example: We'll select the first available record set
record_set_ids = [rs.id for rs in record_sets]
print("Record set ids we'll extract:", record_set_ids)
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for '@id'={record_set_id}: shape={df.shape}")

# Let's inspect columns of the first record set
main_record_set_id = record_set_ids[0]
print(f"Column names in '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
This section demonstrates filtering, normalization, and grouping.

In [ ]:
# Choose a numeric field from the fields list of main_record_set_id, e.g. something like 'Molecular Weight' or 'Abundance'
df = dataframes[main_record_set_id]

# Inspect columns for plausible numeric fields (e.g. 'MW', 'Abundance', 'coverage', etc.)
print("Available columns for EDA:", df.columns.tolist())

# We'll try common names for protein mass spec datasets: 'MW' (molecular weight), 'Coverage_Percent', 'Peptide_Count', etc.
candidate_numeric_fields = [col for col in df.columns if any(key.lower() in col.lower() for key in ["mw", "weight", "coverage", "peptide", "count", "abundance", "score"])]
print("Candidate numeric fields:", candidate_numeric_fields)

# Pick the first candidate numeric field for demonstration
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
    print(f"Using '{numeric_field}' as the numeric field for EDA.")
else:
    raise ValueError("No obvious numeric field found in the columns.")

# Remove rows where field is missing or not numeric
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
filtered_df = df[df[numeric_field] > 0].copy()

# Threshold filtering (example: numeric_field > 1000 or median)
threshold = filtered_df[numeric_field].median()
filtered_above = filtered_df[filtered_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (median): {len(filtered_above)} rows")
print(filtered_above.head())

# Normalize the column (z-score)
filtered_above[f"{numeric_field}_normalized"] = (filtered_above[numeric_field] - filtered_above[numeric_field].mean()) / filtered_above[numeric_field].std()
print(filtered_above[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical field (such as 'Description', 'Gene', or the first suitable string column)
# Find candidate non-numeric fields
candidate_group_fields = [col for col in df.columns if col != numeric_field and df[col].dtype == 'object']
if candidate_group_fields:
    group_field = candidate_group_fields[0]
    print(f"Grouping by '{group_field}'")
    grouped = filtered_above.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    print(grouped.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize the distribution of the numeric field and the effect of filtering and normalization.

In [ ]:
# Plot histogram of the original numeric field
plt.figure(figsize=(8,4))
df[numeric_field].hist(bins=30, color='skyblue', edgecolor='black')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Plot histogram of normalized filtered values
plt.figure(figsize=(8,4))
filtered_above[f'{numeric_field}_normalized'].hist(bins=30, color='salmon', edgecolor='black')
plt.title(f'Normalized Distribution of Filtered {numeric_field}')
plt.xlabel(f'{numeric_field}_normalized')
plt.ylabel('Count')
plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to explore a FAIR-compliant proteomics dataset. 
We performed record set and field introspection, filtered and normalized a numeric field, grouped by a categorical attribute, and visualized data distributions. 

You can now extend this analysis to other fields, record sets, or more advanced visualizations and statistical analyses depending on your research questions.

<br>
*Notebook generated using the Croissant schema at [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)*